# DifIISR — environment setup

Run all cells with a GPU runtime. Stage 1 prepares the pinned official code and Python 3.10. Stage 2 installs the core numerical/GPU dependencies in that isolated environment and tests them. Remaining application dependencies, weights, and inference come later.

The Colab kernel remains unchanged. Runtime files are temporary; rerun this notebook after a reset. GitHub is the canonical notebook source. No tokens or private data are required.


In [ ]:
import sys
import subprocess
from pathlib import Path

def run(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()
    return result.stdout.strip()

print("=== 1. Colab GPU check ===", flush=True)
import torch
print("Notebook Python:", sys.version)
print("Notebook PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))

print("=== 2. Official source ===", flush=True)
repo = Path("/content/DifIISR")
expected_commit = "09ca97ea48d481656dd8090e84099c963059ac41"
if not repo.exists():
    run(["git", "clone", "https://github.com/zirui0625/DifIISR.git", str(repo)])
    run(["git", "checkout", "--detach", expected_commit], cwd=repo)
else:
    if not (repo / ".git").exists():
        raise RuntimeError("Existing /content/DifIISR is not a Git repository; nothing was changed.")
    current = run(["git", "rev-parse", "HEAD"], cwd=repo)
    if current != expected_commit:
        raise RuntimeError("Existing checkout differs from the pinned version; nothing was overwritten.")
status = run(["git", "status", "--porcelain"], cwd=repo)
if status:
    raise RuntimeError("Existing checkout has local changes. Review them before proceeding.")
print("Source commit:", expected_commit)

print("=== 3. Isolated Python environment ===", flush=True)
run([sys.executable, "-m", "pip", "install", "uv"])
run([sys.executable, "-m", "uv", "--version"])
env_dir = Path("/content/difiisr-env")
env_python = env_dir / "bin" / "python"
if not env_dir.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(env_dir)])
if not env_python.exists():
    raise RuntimeError("Existing environment is incomplete; no files were removed.")
run([str(env_python), "-c",
     "import sys; print(sys.version); assert sys.version_info[:2] == (3, 10), 'Expected Python 3.10'"])

print("=== Official requirements (not installed yet) ===", flush=True)
print((repo / "requirements.txt").read_text())
print("SETUP_STAGE_1_OK")
print("Python 3.10 is ready. Proceed to stage 2 below for core dependencies.")
print("The notebook kernel still uses Colab Python; model commands must use:", env_python)


## Stage 2 — core GPU libraries

This downloads several GB and may take several minutes. Uses the official requirements' torch/xformers/scipy versions, paired torchvision, and NumPy 1.25.2 (compatible with SciPy 1.9.3). Does not install packages into the notebook kernel. Success marker: `CORE_GPU_OK`.


In [ ]:
# Stage 2: core dependencies only; no model weights or data downloads.
# This cell requires stage 1 above to succeed.
import os
install_env = os.environ.copy()
install_env.pop("UV_SYSTEM_PYTHON", None)

def run_stage2(args):
    with subprocess.Popen(args, env=install_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        returncode = proc.wait()
    if returncode:
        raise RuntimeError(f"Stage 2 failed (exit {returncode}); see original error above.")

run_stage2([
    sys.executable, "-m", "uv", "pip", "install",
    "--python", str(env_python),
    "torch==2.1.1", "torchvision==0.16.1", "xformers==0.0.23",
    "numpy==1.25.2", "scipy==1.9.3",
])
run_stage2([
    sys.executable, "-m", "uv", "pip", "check",
    "--python", str(env_python),
])
gpu_check = """
import torch, torchvision, xformers, numpy, scipy
from xformers.ops import memory_efficient_attention
print("Isolated torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("xformers:", xformers.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
assert torch.cuda.is_available(), "GPU not available in isolated environment"
print("GPU:", torch.cuda.get_device_name(0))
x = torch.ones((16, 16), device="cuda")
assert torch.allclose(x @ x, torch.full_like(x, 16))
q = torch.randn(1, 32, 1, 32, device="cuda", dtype=torch.float16)
y = memory_efficient_attention(q, q, q)
torch.cuda.synchronize()
assert torch.isfinite(y).all()
print("CORE_GPU_OK")
print("Core libraries passed. Full DifIISR dependencies and inference are not yet verified.")
"""
run_stage2([str(env_python), "-u", "-c", gpu_check])


## Diagnostics only — no installation
Run the following cell in the runtime where the failure occurred. It displays the original error and tests imports separately. Do not reset the runtime first. If the environment is missing, reopen the original session.


In [ ]:
# Read-only diagnostics. Run ONLY this cell in the runtime where stage 2 failed.
import subprocess
from pathlib import Path

diagnostic_python = Path("/content/difiisr-env/bin/python")
diagnostic_code = r'''
import sys, importlib, importlib.metadata, traceback
print("Python:", sys.version, flush=True)
for package in ["torch", "torchvision", "xformers", "numpy", "scipy"]:
    try:
        print("Installed:", package, importlib.metadata.version(package), flush=True)
    except importlib.metadata.PackageNotFoundError:
        print("NOT INSTALLED:", package, flush=True)
for package in ["torch", "torchvision", "xformers", "numpy", "scipy"]:
    print("IMPORT:", package, flush=True)
    try:
        importlib.import_module(package)
        print("IMPORT OK:", package, flush=True)
    except Exception:
        traceback.print_exc()
        sys.exit(1)
import torch
print("CUDA build:", torch.version.cuda, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
try:
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
    x = torch.ones((16, 16), device="cuda")
    assert torch.allclose(x @ x, torch.full_like(x, 16))
    torch.cuda.synchronize()
    print("TORCH_GPU_OK", flush=True)
    from xformers.ops import memory_efficient_attention
    q = torch.randn(1, 32, 1, 32, device="cuda", dtype=torch.float16)
    y = memory_efficient_attention(q, q, q)
    torch.cuda.synchronize()
    assert torch.isfinite(y).all()
    print("CORE_GPU_OK", flush=True)
except Exception:
    traceback.print_exc()
    sys.exit(1)
'''
if not diagnostic_python.exists():
    print("ENVIRONMENT_MISSING: this runtime does not contain the failed environment.")
else:
    result = subprocess.run(
        [str(diagnostic_python), "-u", "-c", diagnostic_code],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    print("=== DIAGNOSTIC START ===")
    print(result.stdout)
    print("Diagnostic exit code:", result.returncode)
    print("=== DIAGNOSTIC END ===")
